# 05 - Ejercicios Prácticos: Metamodelos (Voting y Stacking Classifiers)

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno pondrás en práctica la construcción de comités de votación (*Hard* y *Soft Voting*), la implementación manual y automatizada de *Stacking Out-of-Fold* y la sintonización de hiperparámetros de metamodelos.

---

### Instrucciones
1. Ejecuta la celda de configuración inicial.
2. Completa el código en los bloques marcados con `# TU CODIGO AQUI`.
3. Contrasta tus resultados con el cuaderno `06_Soluciones_Voting_y_Stacking.ipynb`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

np.random.seed(42)
print("Entorno de ejercicios de Metamodelos preparado.")


---
### Ejercicio 1: Cálculo Numérico Manual de Hard Voting vs Soft Voting

Tres clasificadores ($M_1, M_2, M_3$) evalúan una misma observación en un problema binario y devuelven las siguientes probabilidades para la Clase 1 ($P(Y=1 \mid \mathbf{x})$):
- $M_1$: $0.45$ (Predice Clase 0)
- $M_2$: $0.48$ (Predice Clase 0)
- $M_3$: $0.95$ (Predice Clase 1 con alta certidumbre)

1. Calcula manualmente en Python la predicción por **Hard Voting** (mayoría simple sobre las clases predichas $\{0, 0, 1\}$).
2. Calcula la probabilidad promedio y la predicción por **Soft Voting** uniforme.
3. Calcula la probabilidad ponderada y la predicción por **Soft Voting Ponderado** con pesos `w = [1, 1, 2]`.
4. Imprime los tres resultados y explica por qué difieren.


In [ ]:
probs_clase_1 = np.array([0.45, 0.48, 0.95])
pesos = np.array([1, 1, 2])

# TU CODIGO AQUI
# 1. Prediccion Hard Voting
# 2. Probabilidad media y prediccion Soft Voting uniforme
# 3. Probabilidad media ponderada y prediccion con pesos [1, 1, 2]


---
### Ejercicio 2: Comité Heterogéneo con `VotingClassifier` en Dataset Wine

1. Carga `load_wine()` y divide en entrenamiento y prueba ($75\% / 25\%$, estratificado, `random_state=42`).
2. Define tres estimadores heterogéneos:
   - `'lr'`: `make_pipeline(StandardScaler(), LogisticRegression(random_state=42))`
   - `'knn'`: `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`
   - `'rf'`: `RandomForestClassifier(n_estimators=100, random_state=42)`
3. Entrena un `VotingClassifier` con `voting='soft'` e imprime la exactitud de cada modelo individual y la del comité en el conjunto de prueba.


In [ ]:
# TU CODIGO AQUI
# 1. Cargar load_wine y dividir en train/test
# 2. Definir los 3 estimadores y el VotingClassifier(voting='soft')
# 3. Ajustar e imprimir la exactitud comparativa en test


---
### Ejercicio 3: Construcción de un `StackingClassifier` e Inspección del Meta-Clasificador

1. Carga `load_breast_cancer()` y realiza una partición estratificada $75\% / 25\%$ (`random_state=42`).
2. Construye un `StackingClassifier` con los modelos de Nivel 0:
   - `'rf'`: `RandomForestClassifier(n_estimators=100, random_state=42)`
   - `'gb'`: `GradientBoostingClassifier(n_estimators=100, random_state=42)`
   - `'svc'`: `make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))`
   y como `final_estimator` usa `LogisticRegression(random_state=42)` con `cv=5`.
3. Entrena el modelo, muestra la matriz de confusión en prueba e imprime los coeficientes (`stack.final_estimator_.coef_`) asignados a cada modelo base.


In [ ]:
# TU CODIGO AQUI
# 1. Cargar load_breast_cancer y split
# 2. Instanciar y entrenar StackingClassifier(cv=5)
# 3. Imprimir exactitud, coeficientes del final_estimator_ y graficar ConfusionMatrixDisplay


---
### Ejercicio 4: Generación Manual de Meta-Características Out-of-Fold con `cross_val_predict`

Para comprender qué hace `StackingClassifier` por dentro:
1. Utilizando `X_train` y `y_train` de `load_breast_cancer()`, genera las probabilidades *Out-of-Fold* (`cv=5`, `method='predict_proba'`) para `RandomForestClassifier(random_state=42)` y `GradientBoostingClassifier(random_state=42)` usando `cross_val_predict`.
2. Concatena la columna de probabilidad de la clase 1 de ambos modelos para formar la matriz `Z_train` de forma `(N_train, 2)`.
3. Imprime la forma (`shape`) y las primeras 5 filas de `Z_train`.


In [ ]:
# TU CODIGO AQUI
# 1. Usar cross_val_predict(..., cv=5, method='predict_proba') para RF y GB
# 2. Concatenar con np.column_stack las probabilidades de la clase 1
# 3. Imprimir Z_train.shape y las primeras 5 filas


---
### Ejercicio 5: Optimización de Hiperparámetros Anidados en Stacking con `GridSearchCV`

En Scikit-Learn podemos optimizar tanto los modelos de Nivel 0 como el `final_estimator` usando el doble guion bajo (`__`):
1. Sobre el `StackingClassifier` del Ejercicio 3, configura un `GridSearchCV` (`cv=5`, `scoring='accuracy'`) explorando:
   - `'rf__max_depth': [3, None]`
   - `'gb__learning_rate': [0.05, 0.1]`
   - `'final_estimator__C': [0.1, 1.0, 10.0]`
2. Imprime `best_params_` y la exactitud final en el conjunto de prueba.


In [ ]:
# TU CODIGO AQUI
# 1. Definir param_grid para rf__max_depth, gb__learning_rate y final_estimator__C
# 2. Ejecutar GridSearchCV sobre el StackingClassifier
# 3. Imprimir mejores parametros y exactitud en test
